# Kalenjin ASR: Model Training with Wav2Vec2

**Model**: Wav2Vec2-XLS-R-300M
**Task**: Fine-tuning for Kalenjin speech recognition
**Approach**: CTC (Connectionist Temporal Classification)

## Training Strategy:
1. Load pre-trained Wav2Vec2-XLS-R-300M
2. Add CTC head with Kalenjin vocabulary (32 tokens)
3. Fine-tune with SpecAugment
4. Evaluate on validation set
5. Test on held-out test set

## 1. Setup & Dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from datasets import load_from_disk
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer
)
from dataclasses import dataclass
from typing import Dict, List, Union
import json
import numpy as np
from pathlib import Path

print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Dataset & Vocabulary

In [ ]:
# Load preprocessed dataset
dataset = load_from_disk('../processed_data/kalenjin_asr_dataset')

# Load vocabulary
with open('../processed_data/vocab.json', 'r') as f:
    vocab_dict = json.load(f)

print(f"Dataset splits: {list(dataset.keys())}")
print(f"Train: {len(dataset['train'])} samples")
print(f"Validation: {len(dataset['validation'])} samples")
print(f"Test: {len(dataset['test'])} samples")
print(f"\nVocabulary size: {len(vocab_dict)}")

## 3. Create Tokenizer & Processor

In [ ]:
# Create tokenizer from vocabulary
tokenizer = Wav2Vec2CTCTokenizer(
    vocab_dict,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

# Create feature extractor
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

# Combine into processor
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

# Save processor
processor.save_pretrained('../models/wav2vec2-kalenjin/processor')
print("✓ Processor created and saved")

## 4. Prepare Data for Training

In [ ]:
def prepare_dataset(batch):
    """Prepare audio and text for training."""
    # Process audio
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    
    # Process text
    with processor.as_target_processor():
        batch["labels"] = processor(batch["text"]).input_ids
    
    return batch

# Apply preprocessing
print("Preparing dataset...")
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names["train"],
    num_proc=4
)
print("✓ Dataset prepared")

## 5. Data Collator with Padding

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    """Data collator for CTC training."""
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate inputs and labels
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Pad inputs
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Pad labels
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # Replace padding with -100 for loss calculation
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)
print("✓ Data collator created")

## 6. Load Pre-trained Model

In [ ]:
# Load Wav2Vec2-XLS-R-300M
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

# Freeze feature encoder
model.freeze_feature_encoder()

print("✓ Model loaded")
print(f"  Parameters: {model.num_parameters():,}")
print(f"  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Evaluation Metrics

In [ ]:
from evaluate import load

wer_metric = load("wer")
cer_metric = load("cer")

def compute_metrics(pred):
    """Compute WER and CER."""
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

print("✓ Metrics configured")

## 8. Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/wav2vec2-kalenjin",
    group_by_length=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=3e-4,
    warmup_steps=500,
    num_train_epochs=30,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

print("✓ Training configuration:")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  FP16: {training_args.fp16}")

## 9. Initialize Trainer

In [ ]:
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.feature_extractor,
)

print("✓ Trainer initialized")

## 10. Start Training

In [ ]:
print("\n" + "="*70)
print("STARTING TRAINING".center(70))
print("="*70 + "\n")

trainer.train()

print("\n" + "="*70)
print("TRAINING COMPLETE".center(70))
print("="*70)

## 11. Evaluate on Test Set

In [ ]:
print("\nEvaluating on test set...")
test_results = trainer.evaluate(dataset["test"])

print("\n" + "="*70)
print("TEST SET RESULTS".center(70))
print("="*70)
print(f"WER: {test_results['eval_wer']:.4f}")
print(f"CER: {test_results['eval_cer']:.4f}")
print("="*70)

## 12. Save Final Model

In [ ]:
# Save model and processor
model.save_pretrained("../models/wav2vec2-kalenjin/final")
processor.save_pretrained("../models/wav2vec2-kalenjin/final")

# Save training results
with open("../models/wav2vec2-kalenjin/test_results.json", "w") as f:
    json.dump(test_results, f, indent=2)

print("✓ Model saved to: ../models/wav2vec2-kalenjin/final")
print("✓ Training complete!")